In [ ]:
import pandas as pd

# File paths — put this script in the same folder as your files
mapping_file = "main_with_subs_only.xlsx"
indent_file  = "Monthly Indent.xlsx"

# Load data
df_mapping = pd.read_excel(mapping_file)
df_indent   = pd.read_excel(indent_file)

# Rename columns for clarity (using your terms)
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Main_Count': 'Qty_per_Switch',
    'Sub_Count':  'Historical_Total'   # kept but not used in calc
})

df_indent = df_indent.rename(columns={'Part number': 'Switch_Part'})

# Month columns exactly as in your file
month_cols = ["Feb'26", "Mar'26", "Apr'26", "May'26", "Jun'26", "Jul'26"]

# Merge monthly indents into mapping
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

# Calculate daily and 2-days per month (per row)
for month in month_cols:
    daily_col   = f"Daily_{month.replace(\"'\", \"\")}"      # e.g. Daily_Feb26
    twodays_col = f"2Days_{month.replace(\"'\", \"\")}"
    
    df_merged[daily_col]   = (df_merged[month] / 30.0).round(2)
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# ────────────────────────────────────────────────
# 1. Totals per child part (one row per Child_Part)
# ────────────────────────────────────────────────
totals = df_merged.groupby('Child_Part', as_index=False).agg({
    f"Daily_{m.replace(\"'\", \"\")}": 'sum' for m in month_cols
})

# Add 2Days columns
for month in month_cols:
    daily_col   = f"Daily_{month.replace(\"'\", \"\")}"
    twodays_col = f"2Days_{month.replace(\"'\", \"\")}"
    totals[twodays_col] = (totals[daily_col] * 2).round(2)

# Order columns nicely
totals_cols = ['Child_Part'] + \
              [f"Daily_{m.replace(\"'\", \"\")}" for m in month_cols] + \
              [f"2Days_{m.replace(\"'\", \"\")}" for m in month_cols]

totals = totals[totals_cols]

# Save totals file
totals_file = "Child_Totals_2Days_Per_Month.xlsx"
totals.to_excel(totals_file, index=False)
print(f"Totals file saved: {totals_file}  ({len(totals)} rows)")

# ────────────────────────────────────────────────
# 2. Detailed breakdown (one row per child-switch pair)
# ────────────────────────────────────────────────
detailed_cols = ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] + \
                month_cols + \
                [f"Daily_{m.replace(\"'\", \"\")}" for m in month_cols] + \
                [f"2Days_{m.replace(\"'\", \"\")}" for m in month_cols]

detailed = df_merged[detailed_cols]

detailed_file = "Child_Detailed_Breakdown_2Days.xlsx"
detailed.to_excel(detailed_file, index=False)
print(f"Detailed file saved: {detailed_file}  ({len(detailed)} rows)")